In [1]:
import tensorflow
import keras
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image
from matplotlib import pyplot as plt
import kagglehub

In [2]:
path = kagglehub.dataset_download("hongweicao/catanddogsmall")
train_data = path + "/dogvscat_small/train"
valid_data = path + "/dogvscat_small/validation"

Using Colab cache for faster access to the 'catanddogsmall' dataset.


In [3]:
train_datagen = ImageDataGenerator(
    rescale = 1/255,
    rotation_range = 20,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    horizontal_flip = True,
    cval = 225,
    brightness_range = (0.5, 1.5)
)

In [4]:
valid_datagen = ImageDataGenerator(
    rescale = 1/255
)

In [5]:
train_data_generator = train_datagen.flow_from_directory(train_data, target_size = (250,250), batch_size = 30, class_mode="binary")
valid_data_generator = valid_datagen.flow_from_directory(valid_data, target_size = (250,250), batch_size = 30, class_mode="binary")

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


In [6]:
base_model = ResNet50(weights="imagenet",include_top=False,input_shape=(250,250,3))

In [7]:
for layer in base_model.layers:
    layer.trainable = False
x = base_model.output
x = Flatten()(x)
x = Dense(150, activation="relu")(x)
x = Dropout(0.2)(x)
x = Dense(75, activation="relu")(x)
x = Dropout(0.2)(x)
prediction = Dense(1, activation="sigmoid")(x)

model = Model(inputs = base_model.input, outputs = prediction)
model.compile(optimizer = "Adam", loss = "binary_crossentropy",metrics = ["accuracy"])

In [8]:
history = model.fit(train_data_generator,validation_data=valid_data_generator, epochs = 30, verbose = 1)

Epoch 1/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 81s 971ms/step - accuracy: 0.5215 - loss: 2.1091 - val_accuracy: 0.5030 - val_loss: 0.6930
Epoch 2/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 42s 620ms/step - accuracy: 0.4750 - loss: 0.6964 - val_accuracy: 0.5020 - val_loss: 0.6929
Epoch 3/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 41s 618ms/step - accuracy: 0.4965 - loss: 0.7011 - val_accuracy: 0.4980 - val_loss: 0.6929
Epoch 4/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 41s 610ms/step - accuracy: 0.4975 - loss: 0.7057 - val_accuracy: 0.5000 - val_loss: 0.6927
Epoch 5/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 41s 609ms/step - accuracy: 0.5000 - loss: 0.6932 - val_accuracy: 0.4980 - val_loss: 0.6928
Epoch 6/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 41s 612ms/step - accuracy: 0.4845 - loss: 0.6952 - val_accuracy: 0.5010 - val_loss: 0.6928
Epoch 7/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 41s 610ms/step - accuracy: 0.5045 - loss: 0.7294 - val_accuracy: 0.5000 - val_loss: 0.6927
Epoch 8/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 41s 613ms/step - accuracy: 0.5000 - loss: 0.6933 - val_accu

In [10]:
loss, accuracy = model.evaluate(valid_data_generator)
print("Loss: ", loss)
print("Accuracy: ", accuracy)
image_path = path + "/dogvscat_small/test/dogs/1504.jpg"
img = image.load_img(image_path, target_size=(250,250))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array /= 255
prediction = model.predict(img_array)
print("Prediction: ", prediction)
label = "Dog" if prediction[0] > 0.5 else "Cat"
print("Label: ", label)

34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.5000 - loss: 0.6925
Loss:  0.6924550533294678
Accuracy:  0.5
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Prediction:  [[0.50038403]]
Label:  Dog
